In [38]:
import pandas as pd
import json
import ir_datasets
from src.data import DATA_DIR_PROCESSED, DATA_DIR_RAW
import os
from topic_gen.evaluate import QrelsEvaluator, ROSKendallTau, ROSRBO, binarize_qrels, load_runs_from_path
import ir_measures
from pathlib import Path

from topic_gen import logger
logger.setLevel("DEBUG")

## Distinguishability

RQ: How well can the generated topic help to distiguish between systems?


In [14]:
from ir_measures import nDCG, MAP, RBP

runs = load_runs_from_path(DATA_DIR_RAW / "trec-2004-runs")

In [ ]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_PROCESSED / "qrels"

predictions = []
names = []
metadata_records = []
for result in os.listdir(BASE_DIR):

    # metadata
    with open(os.path.join(BASE_DIR, result, "metadata.json")) as f:
        metadata = json.load(f)

    # select only qrels
    if metadata["topics"]["prompt"] in {None, "../data/raw/prompts/trec-contrastive.yaml"}:
        metadata_records.append(metadata)
        # predictions
        qrels = ir_measures.read_trec_qrels(
            os.path.join(BASE_DIR, result, "qrels.csv.gz"))
        predictions.append(qrels)
        names.append(result)

In [26]:
# metadata table
metadata = pd.DataFrame(metadata_records)
metadata = metadata.join(pd.json_normalize(
    metadata["topics"]).add_prefix("topics_"))
metadata.drop(columns=["topics"], inplace=True)
metadata["topics_prompt"] = metadata["topics_prompt"].apply(
    lambda p: str(Path(p).stem) if pd.notnull(p) else "human")
metadata["prompt"] = metadata["prompt"].apply(lambda p: str(Path(p).stem))
metadata["model"] = metadata["model"].str.replace("-MT1000", "")
metadata["model"] = metadata["model"].str.replace("-MT100", "")

In [17]:
reference = binarize_qrels(ir_datasets.load(
    "disks45/nocr/trec-robust-2004").qrels_iter())

In [22]:
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=reference,
    measures=[ROSKendallTau(runs, measures=[nDCG@10, MAP@100]),
              ROSRBO(runs, measures=[nDCG@10, MAP@100], p=0.6, k=100)],
    # bootstrap=20,
    names=names
)

[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 11 / 2940
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 5 / 2946
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 4 / 2947
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 16 / 2864
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 8 / 2943
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 2 / 2949
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] 

In [30]:
table = pd.DataFrame(res)

In [32]:
table = table.pivot(index="name", columns="measure", values="value").round(3)[
    ["ROSKendallTau(nDCG@10)", "ROSRBO(nDCG@10)", "ROSKendallTau(AP@100)", "ROSRBO(AP@100)"]]

In [34]:
table = table.merge(metadata, left_on="name", right_on="date")

In [ ]:
table[["model", "topics_model", "topics_prompt", "topics_nqueries", "topics_ndocsneg", "topics_ndocspos",
       "ROSKendallTau(nDCG@10)", "ROSRBO(nDCG@10)", "ROSKendallTau(AP@100)", "ROSRBO(AP@100)"]]

,model,topics_model,topics_prompt,topics_nqueries,topics_ndocsneg,topics_ndocspos,ROSKendallTau(nDCG@10),ROSRBO(nDCG@10),ROSKendallTau(AP@100),ROSRBO(AP@100)
0,qwen3-30B-no-think,trec assessors,human,NaN,NaN,NaN,0.098,0.978,0.203,0.999
1,qwen3-14B-no-think,trec assessors,human,NaN,NaN,NaN,0.194,0.924,0.157,0.994
2,qwen3-30B-no-think,qwen3-30B-no-think,trec-contrastive,1.0,1.0,1.0,0.081,0.928,0.230,0.997
3,qwen3-30B-no-think,qwen3-30B-no-think,trec-contrastive,3.0,2.0,2.0,0.250,0.923,0.146,0.994
4,qwen3-30B-no-think,qwen3-14B-no-think,trec-contrastive,1.0,1.0,1.0,0.155,0.926,0.175,0.976
5,qwen3-30B-no-think,qwen3-30B-no-think,trec-contrastive,5.0,3.0,3.0,0.263,0.928,0.281,0.994
6,qwen3-30B-no-think,qwen3-14B-no-think,trec-contrastive,3.0,2.0,2.0,0.079,0.928,0.241,0.978
7,qwen3-30B-no-think,qwen3-14B-no-think,trec-contrastive,5.0,3.0,3.0,0.136,0.974,0.148,0.998
8,qwen3-14B-no-think,qwen3-30B-no-think,trec-contrastive,1.0,1.0,1.0,0.143,0.925,0.114,0.992
9,qwen3-14B-no-think,qwen3-30B-no-think,trec-contrastive,3.0,2.0,2.0,0.054,0.921,0.089,0.991
